# D-FINE Model Trainer UI

This dashboard deliberately separates two upstream projects:

- **Official [Peterande/D-FINE](https://github.com/Peterande/D-FINE)**: object detection only; custom datasets use COCO JSON bounding boxes.
- **[ArgoHA/D-FINE-seg](https://github.com/ArgoHA/D-FINE-seg)**: a separate implementation supporting object detection and instance segmentation.

The official detector does **not** consume a YOLO `data.yaml` directly. This notebook validates either a standalone YOLO dataset or a complete Data Labeller export and creates a non-destructive COCO staging view for both official D-FINE and D-FINE-seg, preserving the existing train/validation split without rewriting the labelled source.

## Annotation formats

- YOLO detection row: `class_id x_center y_center width height` (all coordinates normalized to 0–1).
- YOLO instance-segmentation row: `class_id x1 y1 x2 y2 ... xN yN` (normalized polygon points, at least three points).
- COCO detection annotation: pixel `bbox: [x, y, width, height]` plus `category_id`, `image_id`, `area`, and `iscrowd`.
- COCO instance-segmentation annotation: the same box fields plus `segmentation: [[x1, y1, x2, y2, ...]]` in pixel coordinates.

A detection box cannot be used as a segmentation mask: instance segmentation requires the polygon coordinates from `yolo26_segmentation`, not the five-column detection labels.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('ipywidgets') is None:
    print(f'Installing ipywidgets into the active kernel: {sys.executable}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets>=8.1'])
    importlib.invalidate_caches()

candidates = [Path.cwd(), Path.cwd() / 'model_training', Path.cwd().parent]
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidates if (candidate / 'src' / 'naartjie_cv').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the model_training folder or its parent.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

OFFICIAL_DFINE_ROOT = PROJECT_ROOT / 'third_party' / 'D-FINE'
if (OFFICIAL_DFINE_ROOT / 'requirements.txt').is_file():
    required_modules = ('faster_coco_eval', 'tensorboard', 'calflops', 'transformers', 'loguru')
    missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
    if missing_modules:
        print(f'Installing official D-FINE dependencies into: {sys.executable}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(OFFICIAL_DFINE_ROOT / 'requirements.txt')])
        importlib.invalidate_caches()
else:
    print('Official D-FINE checkout not found under third_party/D-FINE; select another checkout in the dashboard.')

DFINE_SEG_ROOT = PROJECT_ROOT / 'third_party' / 'D-FINE-seg'
if (DFINE_SEG_ROOT / 'src' / 'dl' / 'train.py').is_file():
    dfine_seg_requirements = {
        'albumentations': 'albumentations==2.0.8',
        'hydra': 'hydra-core==1.3.2',
        'torchmetrics': 'torchmetrics==1.9.0',
        'wandb': 'wandb==0.26.1',
        'tabulate': 'tabulate>=0.9',
        'huggingface_hub': 'huggingface-hub>=0.30',
    }
    missing_packages = [package for module, package in dfine_seg_requirements.items() if importlib.util.find_spec(module) is None]
    if missing_packages:
        print(f'Installing D-FINE-seg training dependencies into: {sys.executable}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
        importlib.invalidate_caches()
else:
    print('D-FINE-seg checkout not found under third_party/D-FINE-seg; select another checkout in the dashboard.')
print(f'Notebook Python: {sys.executable}')
print(f'Model Trainer root: {PROJECT_ROOT}')
print(f'Official D-FINE root: {OFFICIAL_DFINE_ROOT}')
print(f'D-FINE-seg root: {DFINE_SEG_ROOT}')

## Training dashboard

1. Choose the official detector or a D-FINE-seg task.
2. Select the complete export or standalone YOLO dataset root.
3. Select the matching upstream checkout and its Python environment.
4. Validate, smoke-test the real loader, and prepare a dry run.
5. Confirm and start training.

In [ ]:
import naartjie_cv.dfine_notebook_ui as dfine_notebook_ui

importlib.reload(dfine_notebook_ui)
dashboard = dfine_notebook_ui.create_dfine_training_dashboard(PROJECT_ROOT)
dashboard

## Outputs

Official D-FINE runs are written under `runs/dfine/`. D-FINE-seg runs are written under `runs/dfine-seg/` and contain the generated config, non-destructive prepared dataset view, model checkpoints, logs, and `run_metadata.json`.

## Held-out test evaluation

Run the cell below **once, after training and checkpoint selection are final**. It automatically selects the newest D-FINE run containing `best_stg2.pth`, redirects D-FINE's evaluation loader from `val.json` to the untouched `test.json`, and writes the console metrics plus `eval.pth` under `test_results/`. It never updates the model weights. Set `TEST_RUN_NAME` to a run name when you need to evaluate a specific older run.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Keep this cell runnable after a kernel restart, even if the setup cell was not run.
project_candidates = (Path.cwd(), Path.cwd() / 'model_training', Path.cwd().parent)
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in project_candidates if (candidate / 'src' / 'naartjie_cv').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the model_training folder or its parent.')
OFFICIAL_DFINE_ROOT = PROJECT_ROOT / 'third_party' / 'D-FINE'

TEST_RUN_NAME = None  # None selects the newest valid run; otherwise enter an exact run name.
TEST_DEVICE = '0'  # Use 'cpu' for CPU-only evaluation.
runs_root = PROJECT_ROOT / 'runs' / 'dfine'
if TEST_RUN_NAME is None:
    valid_runs = [
        run for run in runs_root.glob('dfine-n-detect-*')
        if (run / 'best_stg2.pth').is_file()
        and (run / 'dfine_n_training.yml').is_file()
        and (run / 'prepared_dataset' / 'test.json').is_file()
    ]
    if not valid_runs:
        raise FileNotFoundError(f'No completed D-FINE runs with test assets found under {runs_root}')
    TEST_RUN = max(valid_runs, key=lambda run: run.name)
    TEST_RUN_NAME = TEST_RUN.name
else:
    TEST_RUN = runs_root / TEST_RUN_NAME
TEST_CHECKPOINT = TEST_RUN / 'best_stg2.pth'
TEST_CONFIG = TEST_RUN / 'dfine_n_training.yml'
TEST_ANNOTATIONS = TEST_RUN / 'prepared_dataset' / 'test.json'
TEST_OUTPUT = TEST_RUN / 'test_results'
TEST_METRICS_LOG = TEST_OUTPUT / 'test_metrics.txt'

required = (TEST_CHECKPOINT, TEST_CONFIG, TEST_ANNOTATIONS, OFFICIAL_DFINE_ROOT / 'train.py')
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing required test-evaluation files: ' + ', '.join(map(str, missing)))

command = [
    sys.executable,
    str(OFFICIAL_DFINE_ROOT / 'train.py'),
    '--config', str(TEST_CONFIG),
    '--resume', str(TEST_CHECKPOINT),
    '--test-only',
    '--output-dir', str(TEST_OUTPUT),
    '--update', f'val_dataloader.dataset.ann_file={TEST_ANNOTATIONS}',
]
environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
if TEST_DEVICE == 'cpu':
    command.extend(['--device', 'cpu'])
else:
    environment['CUDA_VISIBLE_DEVICES'] = TEST_DEVICE

TEST_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f'Evaluating newest held-out test run: {TEST_RUN_NAME}')
print(subprocess.list2cmdline(command))
with TEST_METRICS_LOG.open('w', encoding='utf-8') as metrics_log:
    process = subprocess.Popen(
        command,
        cwd=OFFICIAL_DFINE_ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        metrics_log.write(line)
        metrics_log.flush()
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'D-FINE test evaluation failed with exit code {return_code}. See {TEST_METRICS_LOG}')
print(f'Test metrics saved to: {TEST_METRICS_LOG}')
print(f'COCO evaluation artifact saved to: {TEST_OUTPUT / "eval.pth"}')

TF Lite

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys

# ============================================================
# SETTINGS
# ============================================================

CHECKPOINT_NAME = "best_stg2.pth"
IMAGE_SIZE = (640, 640)  # height, width
ONNX_OPSET = 16
OVERWRITE = False
RUN_NAME = None  # None selects the newest completed D-FINE run

# ============================================================
# LOCATE PROJECT
# ============================================================

project_candidates = (
    Path.cwd(),
    Path.cwd() / "model_training",
    Path.cwd().parent,
)

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in project_candidates
        if (candidate / "src" / "naartjie_cv").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate model_training. Open the notebook from the "
        "project root or model_training directory."
    )

DFINE_ROOT = PROJECT_ROOT / "third_party" / "D-FINE"
RUNS_ROOT = PROJECT_ROOT / "runs" / "dfine"
EXPORT_SCRIPT = DFINE_ROOT / "tools" / "deployment" / "export_onnx.py"

if not EXPORT_SCRIPT.is_file():
    raise FileNotFoundError(f"D-FINE ONNX exporter not found: {EXPORT_SCRIPT}")

# ============================================================
# SELECT D-FINE RUN
# ============================================================

if RUN_NAME is None:
    valid_runs = [
        run
        for run in RUNS_ROOT.glob("dfine-*-detect-*")
        if (run / CHECKPOINT_NAME).is_file()
        and (run / "dfine_n_training.yml").is_file()
    ]

    if not valid_runs:
        raise FileNotFoundError(
            f"No completed D-FINE runs found under: {RUNS_ROOT}"
        )

    RUN_DIR = max(valid_runs, key=lambda run: run.name)
else:
    RUN_DIR = RUNS_ROOT / RUN_NAME

CHECKPOINT = RUN_DIR / CHECKPOINT_NAME
CONFIG = RUN_DIR / "dfine_n_training.yml"
EXPORT_DIR = RUN_DIR / "exports"
LITERT_DIR = EXPORT_DIR / "litert"

ONNX_PATH = EXPORT_DIR / f"{CHECKPOINT.stem}.onnx"
LITERT_FLOAT32 = LITERT_DIR / f"{CHECKPOINT.stem}_float32.tflite"
LITERT_FLOAT16 = LITERT_DIR / f"{CHECKPOINT.stem}_float16.tflite"

missing_files = [
    path
    for path in (CHECKPOINT, CONFIG, EXPORT_SCRIPT)
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(map(str, missing_files))
    )

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
LITERT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected run: {RUN_DIR.name}")
print(f"Checkpoint: {CHECKPOINT}")

# ============================================================
# INSTALL EXPORT DEPENDENCIES
# ============================================================

required_packages = {
    "onnx": "onnx>=1.17",
    "onnxsim": "onnxsim>=0.4.36",
    "onnx2tf": "onnx2tf",
    "ai_edge_litert": "ai-edge-litert",
}

missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("\nInstalling export dependencies:")
    for package in missing_packages:
        print(f"  {package}")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *missing_packages,
        ]
    )

    importlib.invalidate_caches()

# ============================================================
# EXPORT CHECKPOINT TO ONNX
# ============================================================

height, width = map(int, IMAGE_SIZE)

if ONNX_PATH.is_file() and not OVERWRITE:
    print(f"\nReusing existing ONNX model: {ONNX_PATH}")
else:
    onnx_command = [
        sys.executable,
        str(EXPORT_SCRIPT),
        "--config",
        str(CONFIG),
        "--resume",
        str(CHECKPOINT),
        "--output",
        str(ONNX_PATH),
        "--imgsz",
        str(height),
        str(width),
        "--batch-size",
        "1",
        "--opset",
        str(ONNX_OPSET),
    ]

    print("\nExporting D-FINE checkpoint to ONNX:")
    print(subprocess.list2cmdline(onnx_command))

    subprocess.run(
        onnx_command,
        cwd=DFINE_ROOT,
        check=True,
    )

if not ONNX_PATH.is_file():
    raise FileNotFoundError(
        f"ONNX export did not produce the expected file: {ONNX_PATH}"
    )

# ============================================================
# VERIFY ONNX
# ============================================================

import onnx

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print(f"\nONNX validation passed: {ONNX_PATH}")

# ============================================================
# CONVERT ONNX TO LiteRT / TFLITE
# ============================================================

if LITERT_FLOAT32.is_file() and not OVERWRITE:
    print(f"\nReusing existing LiteRT model: {LITERT_FLOAT32}")
else:
    litert_command = [
        sys.executable,
        "-m",
        "onnx2tf.onnx2tf",
        "-i",
        str(ONNX_PATH),
        "-o",
        str(LITERT_DIR),
        "-tb",
        "flatbuffer_direct",

        # Fix dynamic inputs to deployment batch size 1.
        "-ois",
        f"images:1,3,{height},{width}",
        "orig_target_sizes:1,2",

        # Preserve the NCHW image layout.
        "-k",
        "images",

        # Preserve the second input shape.
        "-kat",
        "orig_target_sizes",

        # Preserve ONNX input/output names.
        "-coion",

        # Compare ONNX and LiteRT outputs.
        "-cotof",

        # Dummy values used for conversion validation.
        "--value_hints",
        "images:0.5",
        f"orig_target_sizes:{height}",

        # Produce an operator-coverage report.
        "--report_op_coverage",
    ]

    print("\nConverting ONNX to LiteRT:")
    print(subprocess.list2cmdline(litert_command))

    subprocess.run(
        litert_command,
        check=True,
    )

if not LITERT_FLOAT32.is_file():
    available_models = list(LITERT_DIR.glob("*.tflite"))

    if not available_models:
        raise FileNotFoundError(
            f"LiteRT conversion produced no .tflite files in: {LITERT_DIR}"
        )

    # Handle converter filename differences.
    float32_candidates = [
        path
        for path in available_models
        if "float32" in path.name.lower()
    ]

    LITERT_FLOAT32 = (
        float32_candidates[0]
        if float32_candidates
        else available_models[0]
    )

# ============================================================
# LiteRT RUNTIME SMOKE TEST
# ============================================================

import numpy as np
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=str(LITERT_FLOAT32))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\nLiteRT inputs:")
for detail in input_details:
    print(
        f"  {detail['name']}: "
        f"shape={tuple(detail['shape'])}, dtype={detail['dtype']}"
    )

print("\nLiteRT outputs:")
for detail in output_details:
    print(
        f"  {detail['name']}: "
        f"shape={tuple(detail['shape'])}, dtype={detail['dtype']}"
    )

# Find the image and original-size inputs.
image_input = next(
    (
        detail
        for detail in input_details
        if len(detail["shape"]) == 4
        or "images" in detail["name"].lower()
    ),
    None,
)

size_input = next(
    (
        detail
        for detail in input_details
        if len(detail["shape"]) == 2
        and detail is not image_input
    ),
    None,
)

if image_input is None:
    raise RuntimeError("Could not identify the LiteRT image input.")

if size_input is None:
    raise RuntimeError("Could not identify the orig_target_sizes input.")

image_shape = tuple(int(value) for value in image_input["shape"])

# Support either preserved NCHW or converted NHWC layout.
if image_shape == (1, 3, height, width):
    test_image = np.random.rand(
        1, 3, height, width
    ).astype(image_input["dtype"])
elif image_shape == (1, height, width, 3):
    test_image = np.random.rand(
        1, height, width, 3
    ).astype(image_input["dtype"])
else:
    raise RuntimeError(
        f"Unexpected LiteRT image input shape: {image_shape}"
    )

target_sizes = np.array(
    [[height, width]],
    dtype=size_input["dtype"],
)

interpreter.set_tensor(image_input["index"], test_image)
interpreter.set_tensor(size_input["index"], target_sizes)
interpreter.invoke()

results = {
    detail["name"]: interpreter.get_tensor(detail["index"])
    for detail in output_details
}

print("\nLiteRT runtime test passed:")
for name, value in results.items():
    print(
        f"  {name}: shape={value.shape}, "
        f"dtype={value.dtype}, "
        f"min={value.min():.6f}, max={value.max():.6f}"
    )

# ============================================================
# FINAL RESULT
# ============================================================

print("\nExport complete")
print(f"Run:            {RUN_DIR.name}")
print(f"ONNX:           {ONNX_PATH}")
print(f"LiteRT float32: {LITERT_FLOAT32}")

if LITERT_FLOAT16.is_file():
    print(f"LiteRT float16: {LITERT_FLOAT16}")

print(f"Reports:        {LITERT_DIR}")

TensorRT

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys
import time

# ============================================================
# SETTINGS
# ============================================================

CHECKPOINT_NAME = "best_stg2.pth"
RUN_NAME = None                 # None selects the newest D-FINE run
IMAGE_SIZE = (640, 640)         # height, width
ONNX_OPSET = 16

USE_FP16 = True
PROFILE_BATCHES = (1, 1, 8)    # minimum, optimum, maximum
TEST_BATCH_SIZE = 1
WORKSPACE_GIB = 2
OVERWRITE = False

# ============================================================
# LOCATE PROJECT
# ============================================================

project_candidates = (
    Path.cwd(),
    Path.cwd() / "model_training",
    Path.cwd().parent,
)

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in project_candidates
        if (candidate / "src" / "naartjie_cv").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate model_training. Open the notebook from the "
        "project root or model_training directory."
    )

DFINE_ROOT = PROJECT_ROOT / "third_party" / "D-FINE"
RUNS_ROOT = PROJECT_ROOT / "runs" / "dfine"
EXPORT_SCRIPT = DFINE_ROOT / "tools" / "deployment" / "export_onnx.py"

# ============================================================
# SELECT RUN
# ============================================================

if RUN_NAME is None:
    valid_runs = [
        run
        for run in RUNS_ROOT.glob("dfine-*-detect-*")
        if (run / CHECKPOINT_NAME).is_file()
        and (run / "dfine_n_training.yml").is_file()
    ]

    if not valid_runs:
        raise FileNotFoundError(
            f"No completed D-FINE runs found under: {RUNS_ROOT}"
        )

    RUN_DIR = max(valid_runs, key=lambda run: run.name)
else:
    RUN_DIR = RUNS_ROOT / RUN_NAME

CHECKPOINT = RUN_DIR / CHECKPOINT_NAME
CONFIG = RUN_DIR / "dfine_n_training.yml"
EXPORT_DIR = RUN_DIR / "exports"

ONNX_PATH = EXPORT_DIR / f"{CHECKPOINT.stem}.onnx"
FP16_ONNX_PATH = EXPORT_DIR / f"{CHECKPOINT.stem}_mixed_fp16.onnx"

precision_name = "fp16" if USE_FP16 else "fp32"

min_batch, opt_batch, max_batch = map(int, PROFILE_BATCHES)

ENGINE_PATH = (
    EXPORT_DIR
    / f"{CHECKPOINT.stem}_{precision_name}_b{min_batch}-{max_batch}.engine"
)

required_files = (CHECKPOINT, CONFIG, EXPORT_SCRIPT)
missing_files = [path for path in required_files if not path.is_file()]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n" + "\n".join(map(str, missing_files))
    )

if not (0 < min_batch <= opt_batch <= max_batch):
    raise ValueError(
        "PROFILE_BATCHES must be ordered as "
        "(minimum, optimum, maximum)."
    )

if not (min_batch <= TEST_BATCH_SIZE <= max_batch):
    raise ValueError(
        "TEST_BATCH_SIZE must be within the optimization profile."
    )

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected run: {RUN_DIR.name}")
print(f"Checkpoint:   {CHECKPOINT}")

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

required_packages = {
    "onnx": "onnx>=1.17",
    "tensorrt": "tensorrt-cu13",
}

if USE_FP16:
    required_packages["modelopt"] = "nvidia-modelopt[onnx]>=0.44"

missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("\nInstalling dependencies:")

    for package in missing_packages:
        print(f"  {package}")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *missing_packages,
        ]
    )

    importlib.invalidate_caches()

import numpy as np
import onnx
import tensorrt as trt
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "TensorRT requires an NVIDIA GPU with CUDA available."
    )

print(f"\nPyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.version.cuda}")
print(f"TensorRT: {trt.__version__}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")

# ============================================================
# EXPORT CHECKPOINT TO ONNX
# ============================================================

height, width = map(int, IMAGE_SIZE)

if ONNX_PATH.is_file() and not OVERWRITE:
    print(f"\nReusing ONNX model: {ONNX_PATH}")
else:
    onnx_command = [
        sys.executable,
        str(EXPORT_SCRIPT),
        "--config",
        str(CONFIG),
        "--resume",
        str(CHECKPOINT),
        "--output",
        str(ONNX_PATH),
        "--imgsz",
        str(height),
        str(width),
        "--batch-size",
        "1",
        "--opset",
        str(ONNX_OPSET),

        # Avoid the WindowsPath incompatibility in onnxsim.
        "--no-simplify",
    ]

    print("\nExporting D-FINE checkpoint to ONNX:")
    print(subprocess.list2cmdline(onnx_command))

    subprocess.run(
        onnx_command,
        cwd=DFINE_ROOT,
        check=True,
    )

if not ONNX_PATH.is_file():
    raise FileNotFoundError(
        f"ONNX export did not produce: {ONNX_PATH}"
    )

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print(f"ONNX validation passed: {ONNX_PATH}")

# ============================================================
# CONVERT ONNX TO MIXED FP16 FOR TENSORRT 11
# ============================================================

TRT_ONNX_PATH = ONNX_PATH

if USE_FP16:
    import modelopt.onnx.autocast as autocast

    TRT_ONNX_PATH = FP16_ONNX_PATH

    if TRT_ONNX_PATH.is_file() and not OVERWRITE:
        print(f"\nReusing mixed FP16 ONNX: {TRT_ONNX_PATH}")
    else:
        print("\nConverting ONNX to mixed FP16 using NVIDIA ModelOpt...")

        mixed_precision_model = autocast.convert_to_mixed_precision(
            onnx_path=str(ONNX_PATH),
            low_precision_type="fp16",
            keep_io_types=True,
        )

        onnx.checker.check_model(mixed_precision_model)
        onnx.save(mixed_precision_model, str(TRT_ONNX_PATH))

        print(f"Mixed FP16 ONNX saved to: {TRT_ONNX_PATH}")

# ============================================================
# BUILD TENSORRT 11 ENGINE
# ============================================================

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
trt.init_libnvinfer_plugins(TRT_LOGGER, "")

if ENGINE_PATH.is_file() and not OVERWRITE:
    print(f"\nReusing TensorRT engine: {ENGINE_PATH}")
else:
    print("\nBuilding TensorRT engine...")
    print(f"Precision:     {precision_name}")
    print(
        f"Batch profile: min={min_batch}, "
        f"opt={opt_batch}, max={max_batch}"
    )
    print(f"Workspace:     {WORKSPACE_GIB} GiB")

    builder = trt.Builder(TRT_LOGGER)

    # TensorRT 11 is always explicit-batch and strongly typed.
    network = builder.create_network(0)
    parser = trt.OnnxParser(network, TRT_LOGGER)

    if not parser.parse(TRT_ONNX_PATH.read_bytes()):
        parser_errors = "\n".join(
            str(parser.get_error(index))
            for index in range(parser.num_errors)
        )

        raise RuntimeError(
            "TensorRT could not parse the ONNX model:\n"
            + parser_errors
        )

    print("\nTensorRT network inputs:")

    network_input_names = set()

    for index in range(network.num_inputs):
        tensor = network.get_input(index)
        network_input_names.add(tensor.name)

        print(
            f"  {tensor.name}: "
            f"shape={tuple(tensor.shape)}, "
            f"dtype={tensor.dtype}"
        )

    print("\nTensorRT network outputs:")

    for index in range(network.num_outputs):
        tensor = network.get_output(index)

        print(
            f"  {tensor.name}: "
            f"shape={tuple(tensor.shape)}, "
            f"dtype={tensor.dtype}"
        )

    expected_inputs = {"images", "orig_target_sizes"}
    missing_inputs = expected_inputs - network_input_names

    if missing_inputs:
        raise RuntimeError(
            f"Expected TensorRT inputs are missing: "
            f"{sorted(missing_inputs)}"
        )

    config = builder.create_builder_config()

    config.set_memory_pool_limit(
        trt.MemoryPoolType.WORKSPACE,
        int(WORKSPACE_GIB * (1 << 30)),
    )

    # TensorRT 11 does not use BuilderFlag.FP16.
    # FP16 precision is encoded in the mixed-precision ONNX model.

    profile = builder.create_optimization_profile()

    image_min = (min_batch, 3, height, width)
    image_opt = (opt_batch, 3, height, width)
    image_max = (max_batch, 3, height, width)

    size_min = (min_batch, 2)
    size_opt = (opt_batch, 2)
    size_max = (max_batch, 2)

    if not profile.set_shape(
        "images",
        image_min,
        image_opt,
        image_max,
    ):
        raise RuntimeError(
            "Failed to configure the images optimization profile."
        )

    if not profile.set_shape(
        "orig_target_sizes",
        size_min,
        size_opt,
        size_max,
    ):
        raise RuntimeError(
            "Failed to configure the orig_target_sizes profile."
        )

    config.add_optimization_profile(profile)

    build_start = time.perf_counter()

    serialized_engine = builder.build_serialized_network(
        network,
        config,
    )

    if serialized_engine is None:
        raise RuntimeError(
            "TensorRT engine build failed. Review the log above."
        )

    ENGINE_PATH.write_bytes(bytes(serialized_engine))

    build_seconds = time.perf_counter() - build_start

    print(f"\nEngine built in {build_seconds:.1f} seconds")
    print(
        f"Engine size: "
        f"{ENGINE_PATH.stat().st_size / 1e6:.2f} MB"
    )
    print(f"Engine saved to: {ENGINE_PATH}")

# ============================================================
# LOAD ENGINE
# ============================================================

runtime = trt.Runtime(TRT_LOGGER)

engine = runtime.deserialize_cuda_engine(
    ENGINE_PATH.read_bytes()
)

if engine is None:
    raise RuntimeError(
        f"Could not deserialize engine: {ENGINE_PATH}"
    )

context = engine.create_execution_context()

if context is None:
    raise RuntimeError(
        "Could not create TensorRT execution context."
    )

# ============================================================
# SET RUNTIME INPUT SHAPES
# ============================================================

if not context.set_input_shape(
    "images",
    (TEST_BATCH_SIZE, 3, height, width),
):
    raise RuntimeError(
        "Could not set the runtime images shape."
    )

if not context.set_input_shape(
    "orig_target_sizes",
    (TEST_BATCH_SIZE, 2),
):
    raise RuntimeError(
        "Could not set the runtime orig_target_sizes shape."
    )

missing_shapes = context.infer_shapes()

if missing_shapes:
    raise RuntimeError(
        "TensorRT still requires shapes for: "
        + ", ".join(missing_shapes)
    )

# ============================================================
# MAP TENSORRT DTYPES TO PYTORCH
# ============================================================

numpy_to_torch_dtype = {
    np.dtype(np.float32): torch.float32,
    np.dtype(np.float16): torch.float16,
    np.dtype(np.int64): torch.int64,
    np.dtype(np.int32): torch.int32,
    np.dtype(np.int8): torch.int8,
    np.dtype(np.uint8): torch.uint8,
    np.dtype(np.bool_): torch.bool,
}

# ============================================================
# ALLOCATE CUDA TENSORS
# ============================================================

tensors = {}
stream = torch.cuda.Stream()

with torch.cuda.stream(stream):
    for index in range(engine.num_io_tensors):
        name = engine.get_tensor_name(index)
        mode = engine.get_tensor_mode(name)
        trt_dtype = engine.get_tensor_dtype(name)
        numpy_dtype = np.dtype(trt.nptype(trt_dtype))

        if numpy_dtype not in numpy_to_torch_dtype:
            raise TypeError(
                f"Unsupported TensorRT dtype for {name}: "
                f"{trt_dtype}"
            )

        torch_dtype = numpy_to_torch_dtype[numpy_dtype]

        shape = tuple(
            int(value)
            for value in context.get_tensor_shape(name)
        )

        if any(value < 0 for value in shape):
            raise RuntimeError(
                f"Tensor {name} still has a dynamic shape: {shape}"
            )

        if mode == trt.TensorIOMode.INPUT:
            if name == "images":
                tensor = torch.rand(
                    shape,
                    dtype=torch_dtype,
                    device="cuda",
                )

            elif name == "orig_target_sizes":
                tensor = torch.tensor(
                    [[height, width]] * TEST_BATCH_SIZE,
                    dtype=torch_dtype,
                    device="cuda",
                )

            else:
                raise RuntimeError(
                    f"Unknown TensorRT input: {name}"
                )

        else:
            tensor = torch.empty(
                shape,
                dtype=torch_dtype,
                device="cuda",
            )

        tensors[name] = tensor

        address_set = context.set_tensor_address(
            name,
            int(tensor.data_ptr()),
        )

        if not address_set:
            raise RuntimeError(
                f"Could not set the address for tensor: {name}"
            )

    # ========================================================
    # RUN INFERENCE
    # ========================================================

    inference_start = time.perf_counter()

    success = context.execute_async_v3(
        stream_handle=stream.cuda_stream
    )

    if not success:
        raise RuntimeError(
            "TensorRT inference execution failed."
        )

stream.synchronize()

inference_ms = (
    time.perf_counter() - inference_start
) * 1000.0

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\nTensorRT runtime test passed")
print(f"Inference time: {inference_ms:.2f} ms")

print("\nTensorRT tensors:")

for index in range(engine.num_io_tensors):
    name = engine.get_tensor_name(index)
    mode = engine.get_tensor_mode(name)
    tensor = tensors[name]

    mode_name = (
        "INPUT"
        if mode == trt.TensorIOMode.INPUT
        else "OUTPUT"
    )

    print(
        f"  {mode_name:6} {name}: "
        f"shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}"
    )

    if mode == trt.TensorIOMode.OUTPUT:
        output = tensor.detach().cpu()

        print(
            f"         min={output.min().item():.6f}, "
            f"max={output.max().item():.6f}"
        )

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\nExport complete")
print(f"Run:            {RUN_DIR.name}")
print(f"Original ONNX:  {ONNX_PATH}")
print(f"TensorRT ONNX:  {TRT_ONNX_PATH}")
print(f"TensorRT engine:{ENGINE_PATH}")
print(
    "\nThe .engine file is specific to this GPU, CUDA version, "
    "TensorRT version, and batch profile."
)